# Relearning: Newly Resurfaced People (excluding already-leaked)

This notebook shows leakage from relearning **after removing** people who were already leaked by the unlearned model (before relearning).

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize

OUTPUT_DIR = os.path.join(os.environ['HOME'], 'OLMoBenchOutputs', 'saves', 'Train_95frozen_FullSubset')

FIELD_FILE_MAP = {
    'Birth City': 'Birth_City',
    'Email Address': 'Email_Address',
    'Phone Number': 'Phone_Number',
    "Driver's License": 'Drivers_License',
}

METHODS = ['AlphaEdit', 'MemFlex', 'SimNPO', 'GradDiff_OracleGrad']
NUM_ATTEMPTS = 200

METHOD_COLORS = {
    'AlphaEdit':  '#4E79A7',
    'MemFlex':    '#F28E2B',
    'OracleGrad': '#59A14F',
    'SimNPO':     '#E15759',
}
METHOD_ORDER = ['MemFlex', 'AlphaEdit', 'SimNPO', 'OracleGrad']

TRUE_BLACK = '#000000'
LW = 1.5
FS = 20

def load_cached(cache_dir, filename):
    path = os.path.join(cache_dir, filename)
    if not os.path.exists(path):
        print(f'  [MISSING] {filename}')
        return None, None
    with open(path, 'rb') as f:
        cached = pickle.load(f)
    return cached['results'], cached['metrics']

In [ ]:
# Load data and compute newly-resurfaced people per (field, method)
# "Newly resurfaced" = leaked after relearning BUT NOT already leaked by the unlearned model

all_rows = []

for target_field, field_file in FIELD_FILE_MAP.items():
    cache_dir = os.path.join(OUTPUT_DIR, 'cached_notebook_files', field_file)
    if not os.path.exists(cache_dir):
        print(f'[SKIP] {target_field} — cache dir missing')
        continue

    for method in METHODS:
        label = method.replace('GradDiff_OracleGrad', 'OracleGrad')

        # Relearning results (memorized people)
        relearn_results, relearn_metrics = load_cached(
            cache_dir, f'results_relearn_memorized_{method}.pkl')
        if relearn_results is None:
            continue

        # Unlearned baseline results (sentence + QA)
        unlearn_sent_results, _ = load_cached(
            cache_dir, f'unlearned_baseline_sentence_{method}.pkl')
        unlearn_qa_results, _ = load_cached(
            cache_dir, f'unlearned_baseline_qa_{method}.pkl')

        # People leaked after relearning
        relearn_leaked = set(
            relearn_results[relearn_results['leaked']]['person_id'].unique())

        # People already leaked by unlearned model (union of sentence + QA)
        already_leaked = set()
        if unlearn_sent_results is not None:
            already_leaked |= set(
                unlearn_sent_results[unlearn_sent_results['leaked']]['person_id'].unique())
        if unlearn_qa_results is not None:
            already_leaked |= set(
                unlearn_qa_results[unlearn_qa_results['leaked']]['person_id'].unique())

        newly_resurfaced = relearn_leaked - already_leaked
        total_people = relearn_results['person_id'].nunique()

        all_rows.append({
            'Field': target_field,
            'Method': label,
            'Relearn Leaked': len(relearn_leaked),
            'Already Leaked (unlearned)': len(already_leaked),
            'Newly Resurfaced': len(newly_resurfaced),
            'Total People': total_people,
            'Relearn %': 100 * len(relearn_leaked) / total_people,
            'New %': 100 * len(newly_resurfaced) / total_people,
        })

        print(f'{target_field} | {label:12s} | '
              f'relearn={len(relearn_leaked):3d}, '
              f'already={len(already_leaked):3d}, '
              f'new={len(newly_resurfaced):3d} / {total_people}')

df = pd.DataFrame(all_rows)
display(df)

In [ ]:
# Visualize: bar chart per field — newly resurfaced people only

for target_field in FIELD_FILE_MAP.keys():
    field_df = df[df['Field'] == target_field].copy()
    if field_df.empty:
        continue

    bar_order = [m for m in METHOD_ORDER if m in field_df['Method'].values]
    field_df = field_df.set_index('Method').loc[bar_order].reset_index()
    n = len(field_df)
    x = np.arange(n)

    fig, ax = plt.subplots(figsize=(max(8, 2.5 * n), 6))

    ax.bar(x, field_df['New %'], 0.6,
           color=[METHOD_COLORS[m] for m in field_df['Method']],
           edgecolor=TRUE_BLACK, linewidth=LW, zorder=3)

    for i, v in enumerate(field_df['New %']):
        ax.text(i, v + 1, f'{v:.0f}%', ha='center', va='bottom',
                fontsize=FS - 4, fontweight='bold', color=TRUE_BLACK)

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(0, max(field_df['New %'].max() * 1.2, 10))
    ax.set_xticks(x)
    ax.set_xticklabels(field_df['Method'], fontsize=FS, fontweight='bold')
    ax.set_ylabel(f'% Newly Resurfaced (of {field_df["Total People"].iloc[0]})',
                  fontsize=FS - 2, fontweight='bold')
    ax.set_title(f'1B — {target_field}', fontsize=FS + 2, fontweight='bold', pad=12)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_linewidth(LW)
    ax.tick_params(axis='y', labelsize=FS - 4, width=LW, length=6)
    ax.tick_params(axis='x', width=LW, length=6, pad=4)
    ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=LW, zorder=0)

    fig.tight_layout()
    plt.show()

    for _, row in field_df.iterrows():
        print(f"  {row['Method']:12s}: "
              f"{int(row['Newly Resurfaced'])} newly resurfaced / "
              f"{int(row['Total People'])} total "
              f"(excluded {int(row['Already Leaked (unlearned)'])} already leaked)")

In [ ]:
# Jaccard overlap heatmaps on newly-resurfaced people only

HEATMAP_FILL = 0.92
HEATMAP_BORDER = 1.5

for target_field, field_file in FIELD_FILE_MAP.items():
    cache_dir = os.path.join(OUTPUT_DIR, 'cached_notebook_files', field_file)
    if not os.path.exists(cache_dir):
        continue

    # Rebuild newly-resurfaced sets per method
    new_leaked = {}
    for method in METHODS:
        label = method.replace('GradDiff_OracleGrad', 'OracleGrad')

        relearn_results, _ = load_cached(cache_dir, f'results_relearn_memorized_{method}.pkl')
        if relearn_results is None:
            continue
        unlearn_sent, _ = load_cached(cache_dir, f'unlearned_baseline_sentence_{method}.pkl')
        unlearn_qa, _ = load_cached(cache_dir, f'unlearned_baseline_qa_{method}.pkl')

        relearn_set = set(relearn_results[relearn_results['leaked']]['person_id'].unique())
        already = set()
        if unlearn_sent is not None:
            already |= set(unlearn_sent[unlearn_sent['leaked']]['person_id'].unique())
        if unlearn_qa is not None:
            already |= set(unlearn_qa[unlearn_qa['leaked']]['person_id'].unique())

        new_leaked[label] = relearn_set - already

    method_names = [m for m in METHOD_ORDER if m in new_leaked]
    n = len(method_names)
    if n < 2:
        continue

    # Jaccard matrix
    mask_upper = np.tril(np.ones((n, n), dtype=bool), k=-1)
    jac = np.zeros((n, n))
    for i, m1 in enumerate(method_names):
        for j, m2 in enumerate(method_names):
            u = len(new_leaked[m1] | new_leaked[m2])
            jac[i, j] = len(new_leaked[m1] & new_leaked[m2]) / u if u > 0 else 0

    fig, ax = plt.subplots(figsize=(8, 7))
    cmap_obj = plt.get_cmap('BuPu')
    norm = Normalize(vmin=0, vmax=1)

    ax.set_facecolor('white')
    ax.set_xlim(0, n)
    ax.set_ylim(n, 0)

    gap_frac = 1 - HEATMAP_FILL
    pad = gap_frac / 2

    for i in range(n):
        for j in range(n):
            if not mask_upper[i, j]:
                val = jac[i, j]
                color = cmap_obj(norm(val))
                ax.add_patch(plt.Rectangle(
                    (j, i), 1, 1, fill=True, facecolor='white', edgecolor='none', zorder=2))
                ax.add_patch(plt.Rectangle(
                    (j + pad, i + pad), 1 - 2*pad, 1 - 2*pad,
                    fill=True, facecolor=color, edgecolor=TRUE_BLACK,
                    linewidth=HEATMAP_BORDER, zorder=3))
                lum = 0.299*color[0] + 0.587*color[1] + 0.114*color[2]
                txt_c = 'white' if lum < 0.5 else '#333333'
                ax.text(j + 0.5, i + 0.5, f'{val:.2f}',
                        ha='center', va='center', fontsize=FS,
                        fontweight='bold', color=txt_c, zorder=4)

    ax.set_xticks([k + 0.5 for k in range(n)])
    ax.set_xticklabels(method_names, fontsize=FS - 2, fontweight='bold')
    ax.set_yticks([k + 0.5 for k in range(n)])
    ax.set_yticklabels(method_names, fontsize=FS - 2, fontweight='bold')
    ax.tick_params(axis='both', which='both', length=0, pad=6)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(f'1B — {target_field}\nJaccard overlap (newly resurfaced only)',
                 fontsize=FS, fontweight='bold', pad=12)

    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=FS - 6)

    fig.tight_layout()
    plt.show()

    for m in method_names:
        print(f'  {m}: {len(new_leaked[m])} newly resurfaced')

In [ ]:
# Summary: grouped bar chart across all fields (newly resurfaced % only)

if not df.empty:
    fields_present = [f for f in FIELD_FILE_MAP.keys() if f in df['Field'].values]
    methods_present = [m for m in METHOD_ORDER if m in df['Method'].values]
    n_fields = len(fields_present)
    n_methods = len(methods_present)

    bar_w = 0.18
    x = np.arange(n_fields)

    fig, ax = plt.subplots(figsize=(max(12, 3.5 * n_fields), 7))

    for j, method in enumerate(methods_present):
        offset = (j - (n_methods - 1) / 2) * bar_w
        vals = []
        for field in fields_present:
            row = df[(df['Field'] == field) & (df['Method'] == method)]
            vals.append(row['Newly Resurfaced'].values[0] if len(row) > 0 else 0)
        ax.bar(x + offset, vals, bar_w,
               color=METHOD_COLORS[method], edgecolor=TRUE_BLACK,
               linewidth=LW, label=method, zorder=3)
        for i, v in enumerate(vals):
            if v > 0:
                ax.text(x[i] + offset, v + 0.5, str(v), ha='center', va='bottom',
                        fontsize=13, fontweight='bold', color=TRUE_BLACK)

    ax.set_xticks(x)
    ax.set_xticklabels(fields_present, fontsize=FS - 2, fontweight='bold')
    ax.set_ylabel('# Newly Resurfaced People', fontsize=FS, fontweight='bold', labelpad=12)
    ax.set_title('1B — Newly Resurfaced People per Method & Field\n(excluding already leaked by unlearned model)',
                 fontsize=FS, fontweight='bold', pad=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_linewidth(LW)
    ax.tick_params(axis='y', labelsize=FS - 4, width=LW, length=6)
    ax.tick_params(axis='x', width=LW, length=6, pad=4)
    ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=LW, zorder=0)
    ax.legend(fontsize=FS - 6, frameon=False, loc='upper right')
    fig.tight_layout()
    plt.show()

    # Pivot table
    pivot = df.pivot(index='Method', columns='Field', values='Newly Resurfaced')
    pivot = pivot.reindex(index=methods_present, columns=fields_present)
    display(pivot)